[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-03-text-preprocessing.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 3 — Cleaning Text: Tokenize, Normalize, Stem, Remove Stop Words

**MSIS · AME 5053 · Week 3 · 3 hours**

Before you can count words, search text, or train any model, you have to **clean the
text up** — turn a raw string into a tidy list of units you can work with. That cleaning
is called **preprocessing**, and almost every NLP system starts with it.

Today you build the standard preprocessing steps, in order:

1. **Tokenize** — split text into words (harder than it looks)
2. **Normalize** — case, Unicode, punctuation
3. **Stem** and **lemmatize** — cut words back to a root form
4. **Remove stop words** — drop the very common words

And, as always, you will see where each step **helps** and where it quietly **destroys
your meaning** — because the biggest mistake in preprocessing is doing a step you should
have skipped.

**Tools today:** **NLTK** and **spaCy**, the two workhorse libraries. You met the idea of
tokens in session 3 and stemming in session 5; today you run them for real.

Work through every cell, in order.

---
## Part 0 — Setup

Run these. The first installs the libraries; the second downloads the data files NLTK
needs (word lists, the tokenizer model). **Both run every session** — Colab forgets
everything between sessions.

In [ ]:
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm -q
print("Done.")

In [ ]:
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

import spacy
nlp = spacy.load("en_core_web_sm")
print("NLTK data and spaCy model ready.")

> **Save your own copy now:** File → Save a copy in Drive.

Here is our text for the day — four short product reviews. Run it.

In [ ]:
reviews = [
    "This phone is absolutely AMAZING! The battery life is not bad at all.",
    "I don't like the camera. Photos are blurry and colours look washed out.",
    "Best purchase ever. These studies show it lasts longer than the competitors.",
    "The screen cracked in a week. Terrible quality, would not recommend.",
]
for r in reviews:
    print("-", r)

# Verified output:
#   - This phone is absolutely AMAZING! The battery life is not bad at all.
#   - I don't like the camera. Photos are blurry and colours look washed out.
#   - Best purchase ever. These studies show it lasts longer than the competitors.
#   - The screen cracked in a week. Terrible quality, would not recommend.

---
## Part 1 — Tokenizing: splitting text into words

A **token** is one unit of text — usually a word, but also a number or a punctuation
mark. Tokenizing is step one of everything.

Your first instinct is `text.split()`. It splits on spaces. Watch why that is not enough.

In [ ]:
text = "I don't like it, really!"

print("naive split():", text.split())

# Verified output:
#   naive split(): ['I', "don't", 'like', 'it,', 'really!']

Two problems, both from splitting on spaces only:

- `it,` and `really!` keep their punctuation stuck on. Now `it,` and `it` count as
  different words.
- `don't` stays in one lump. Is that one token, or is it `do` + `not`?

A real tokenizer handles both. Here is NLTK's.

In [ ]:
from nltk.tokenize import word_tokenize

print(word_tokenize("I don't like it, really!"))

# Verified output:
#   ['I', 'do', "n't", 'like', 'it', ',', 'really', '!']

Look what it did: punctuation (`,` and `!`) became its own tokens, and `don't` split into
`do` and `n't` — pulling the hidden "not" out into the open. That `n't` matters enormously
for sentiment, and you would have lost it with `split()`.

spaCy tokenizes the same way (it splits `don't` into `do` + `n't` too), and gives you a lot
more per token — which we use later.

In [ ]:
doc = nlp("I don't like it, really!")
print([t.text for t in doc])

# Verified output:
#   ['I', 'do', "n't", 'like', 'it', ',', 'really', '!']

### Exercise 1

Tokenize the **second review** three ways: with `split()`, with NLTK, and with spaCy.
Print the token count for each. Notice they disagree — and that "how many words is this?"
does not have one answer until you choose a tokenizer.

In [ ]:
review = reviews[1]   # "I don't like the camera. ..."

# YOUR CODE HERE
# 1. split()
# 2. nltk word_tokenize
# 3. spaCy  [t.text for t in nlp(review)]
# print len() of each

---
## Part 2 — Splitting into sentences

Sometimes you need sentences, not words. Your instinct is "split on the full stop". That
breaks immediately, because full stops are not only sentence ends — they are in `Mr.`,
`Rs.5`, `U.S.A.` and more.

In [ ]:
from nltk.tokenize import sent_tokenize

text = "Mr. Rao paid Rs.5 for it. He was not happy. It broke."

print("naive split on '.':")
for s in text.split("."):
    print("   ", repr(s))

print()
print("nltk sent_tokenize:")
for s in sent_tokenize(text):
    print("   ", repr(s))

# Verified output:
#   naive split on '.':
#       'Mr'
#       ' Rao paid Rs'
#       '5 for it'
#       ' He was not happy'
#       ' It broke'
#       ''
#   
#   nltk sent_tokenize:
#       'Mr. Rao paid Rs.5 for it.'
#       'He was not happy.'
#       'It broke.'

Splitting on `.` cuts `Mr.` and `Rs.5` in the wrong places and gives you broken fragments.
`sent_tokenize` knows `Mr.` and `Rs.` are abbreviations, not sentence ends, and gets the
three real sentences right. Use the tool; do not split on the dot.

---
## Part 3 — Normalizing: making "the same" word actually the same

`AMAZING`, `Amazing` and `amazing` are the same word to you. To a computer they are three
different strings. **Normalization** collapses these harmless differences so they count as
one.

### Case folding

The commonest step: lowercase everything.

In [ ]:
tokens = ["AMAZING", "Amazing", "amazing", "The", "the"]
print([t.lower() for t in tokens])
print("distinct before:", len(set(tokens)))
print("distinct after :", len(set(t.lower() for t in tokens)))

# Verified output:
#   ['amazing', 'amazing', 'amazing', 'the', 'the']
#   distinct before: 5
#   distinct after : 2

Five strings became three. `amazing` and `the` now each count once, which is what you
want for counting and search.

(Case is not *always* junk — `US` the country vs `us` the word, `Apple` vs `apple`. For
most Unit I work you lowercase anyway; just know you are throwing something away.)

### A subtler trap: Unicode

Two strings can look **identical on screen** and still not be equal, because Unicode
sometimes has more than one way to store the same character. Watch.

In [ ]:
import unicodedata

a = "caf\u00e9"     # café, é as ONE char (U+00E9) -- the "NFC" form
b = "cafe\u0301"    # café, e + a combining accent (U+0301) -- the "NFD" form

print("look identical:", a, b)
print("equal?", a == b)
print("lengths:", len(a), len(b))

# Verified output:
#   look identical: café café
#   equal? False
#   lengths: 4 5

They print the same and they are not equal — one stores `é` as a single character, the
other as `e` plus a floating accent. If a user types one and your data has the other,
your lookup silently misses.

The fix is **Unicode normalization**: force both into the same standard form with
`unicodedata.normalize`. `"NFC"` is the usual choice.

In [ ]:
import unicodedata

a = "caf\u00e9"     # café, é as ONE char (U+00E9) -- the "NFC" form
b = "cafe\u0301"    # café, e + a combining accent (U+0301) -- the "NFD" form

a2 = unicodedata.normalize("NFC", a)
b2 = unicodedata.normalize("NFC", b)
print("equal after NFC?", a2 == b2)
print("lengths:", len(a2), len(b2))

# Verified output:
#   equal after NFC? True
#   lengths: 4 4

This is not a Western-language curiosity. Indian scripts are full of combining marks, so
the same Kannada or Devanagari word can arrive in two encodings that look identical and
compare unequal. **Normalize text to NFC as it comes in**, and this whole class of silent
bug disappears.

(This is the same family of problem as the `regex` vs `re` issue from Lab 2: Indian text
has structure that naive handling gets wrong. Rule of thumb: `regex` for splitting, `NFC`
for comparing.)

### Removing punctuation

Often you want just the words, no punctuation tokens. `regex` (not `re`!) does it cleanly.

In [ ]:
import regex

text = "This phone is absolutely AMAZING!"
# keep only word-characters and spaces; \p{P} would be "punctuation"
cleaned = regex.sub(r"[^\w\s]", "", text)
print(cleaned)
print(cleaned.lower().split())

# Verified output:
#   This phone is absolutely AMAZING
#   ['this', 'phone', 'is', 'absolutely', 'amazing']

---
## Part 4 — Stemming: chopping words to a root

`study`, `studies`, `studying`, `studied` are really one idea. For search and counting you
often want them to collapse into one form. **Stemming** does this the crude way: it chops
off endings by rule. The classic is the **Porter stemmer** (session 5).

In [ ]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

for w in ["study", "studies", "studying", "studied"]:
    print(f"{w:10} -> {ps.stem(w)}")

# Verified output:
#   study      -> studi
#   studies    -> studi
#   studying   -> studi
#   studied    -> studi

All four collapse to `studi`. Notice `studi` is **not a real word** — and that is fine,
because the only job is that related words land on the *same* string. For search, that is
exactly what you want: someone searching `studying` finds a document that said `studies`.

But "chop by rule" is genuinely crude. Watch it on harder words.

In [ ]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

for w in ["organization", "universal", "was", "mice", "better"]:
    print(f"{w:14} -> {ps.stem(w)}")

# Verified output:
#   organization   -> organ
#   universal      -> univers
#   was            -> wa
#   mice           -> mice
#   better         -> better

Look at what happened:

- `organization -> organ` — it changed the meaning entirely.
- `was -> wa` — gibberish.
- `mice -> mice` — it can only chop endings, so an irregular plural defeats it. It cannot
  turn `mice` into `mouse`.
- `better -> better` — same, it has no idea `better` relates to `good`.

Stemming is fast and often good enough for search. But it makes non-words and mangles
irregulars. When you need a *real* root word, you need the next tool.

---
## Part 5 — Lemmatization: the root that is a real word

**Lemmatization** also reduces a word to a root — but a real dictionary root, called the
**lemma**. `mice -> mouse`, `was -> be`, `better -> good`. It uses a dictionary, not
scissors.

There is a catch, and it is the whole lesson of this part: **lemmatization needs to know
the part of speech.** `better` is `good` as an adjective, but `better` can also be a verb
("to better yourself"). Without the part of speech, the lemmatizer guesses "noun" and gets
it wrong.

In [ ]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

# Default: no part of speech given -> it assumes NOUN, and often fails
print("no POS given (assumes noun):")
for w in ["was", "better", "running", "mice"]:
    print(f"   {w:10} -> {wnl.lemmatize(w)}")

# Tell it the real part of speech:  v=verb, a=adjective, n=noun
print("with the right POS:")
print(f"   was     (verb)      -> {wnl.lemmatize('was', pos='v')}")
print(f"   better  (adjective) -> {wnl.lemmatize('better', pos='a')}")
print(f"   running (verb)      -> {wnl.lemmatize('running', pos='v')}")
print(f"   mice    (noun)      -> {wnl.lemmatize('mice', pos='n')}")

# Verified output:
#   no POS given (assumes noun):
#      was        -> wa
#      better     -> better
#      running    -> running
#      mice       -> mouse
#   with the right POS:
#      was     (verb)      -> be
#      better  (adjective) -> good
#      running (verb)      -> run
#      mice    (noun)      -> mouse

With the wrong (default) assumption, `was -> wa` and `better -> better` — no better than
the stemmer. With the correct part of speech, `was -> be`, `better -> good`,
`running -> run`, `mice -> mouse`. All real words.

So who supplies the part of speech? You do not want to do it by hand. **spaCy figures out
the part of speech for you** and lemmatizes in one step. This is why spaCy is the easy
choice for real work.

In [ ]:
doc = nlp("The mice were running and studies were better than before")
for t in doc:
    print(f"   {t.text:10} POS={t.pos_:6} lemma={t.lemma_}")

# Verified output:
#      The        POS=DET    lemma=the
#      mice       POS=NOUN   lemma=mouse
#      were       POS=AUX    lemma=be
#      running    POS=VERB   lemma=run
#      and        POS=CCONJ  lemma=and
#      studies    POS=NOUN   lemma=study
#      were       POS=AUX    lemma=be
#      better     POS=ADJ    lemma=well
#      than       POS=ADP    lemma=than
#      before     POS=ADP    lemma=before

spaCy tagged each word's part of speech and lemmatized correctly — `were -> be`,
`running -> run`, `studies -> study` — with no work from you.

### Stemming vs lemmatization — side by side

Run the comparison. This table is the thing to remember from today.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
ps = PorterStemmer()

words = ["studies", "organization", "was", "mice", "better"]

print(f"{'word':14}{'stem (Porter)':18}{'lemma (spaCy)'}")
print("-" * 46)
for w in words:
    stem = ps.stem(w)
    lemma = nlp(w)[0].lemma_
    print(f"{w:14}{stem:18}{lemma}")

# Verified output:
#   word          stem (Porter)     lemma (spaCy)
#   ----------------------------------------------
#   studies       studi             study
#   organization  organ             organization
#   was           wa                be
#   mice          mice              mouse
#   better        better            well

---
## Part 6 — Stop words, and the mistake everyone makes

**Stop words** are the very common words — `the`, `is`, `a`, `of` — that appear
everywhere and often carry little meaning on their own. For search and topic-finding,
people routinely remove them so the *content* words stand out. NLTK ships a list.

In [ ]:
from nltk.corpus import stopwords
sw = set(stopwords.words("english"))

print("how many stop words:", len(sw))
print("some of them:", sorted(list(sw))[:15])

# Verified output:
#   how many stop words: 198
#   some of them: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
sw = set(stopwords.words("english"))

text = "This is one of the best phones that I have ever bought"
tokens = word_tokenize(text.lower())
kept = [t for t in tokens if t not in sw]

print("before:", tokens)
print("after :", kept)

# Verified output:
#   before: ['this', 'is', 'one', 'of', 'the', 'best', 'phones', 'that', 'i', 'have', 'ever', 'bought']
#   after : ['one', 'best', 'phones', 'ever', 'bought']

`This is one of the ... that I have ever` fell away, leaving `best`, `phones`, `bought` —
the words that actually say what the review is about. For search, that is a genuine win.

### Now the mistake

Look closely at what is *in* that stop-word list.

In [ ]:
from nltk.corpus import stopwords
sw = set(stopwords.words("english"))

for w in ["not", "no", "nor", "against", "only", "very"]:
    print(f"   '{w}' is a stop word: {w in sw}")

# Verified output:
#      'not' is a stop word: True
#      'no' is a stop word: True
#      'nor' is a stop word: True
#      'against' is a stop word: True
#      'only' is a stop word: True
#      'very' is a stop word: True

**`not` is in the list.** And `no`, `nor`, `against`... the very words that carry
*negation*. Watch what happens when you strip stop words from a review.

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
sw = set(stopwords.words("english"))

review = "would not recommend"
kept = [t for t in word_tokenize(review) if t not in sw]

print("original :", review)
print("after stop-word removal:", " ".join(kept))

# Verified output:
#   original : would not recommend
#   after stop-word removal: would recommend

**`would not recommend` became `would recommend`.** Removing stop words flipped the meaning
into its exact opposite. A sentiment model fed this would learn that an angry review is a
happy one.

This is the callback to Lab 1: VADER got `not bad` right *because* it kept the `not`. Throw
the `not` away here and you destroy the sentiment.

**The lesson:** stop-word removal is not free, and not always right.

- **Helps** when you want topics or keywords, and function words are noise — search, IR
  (that is lab 4). Remove away.
- **Hurts** when meaning depends on those words — sentiment, question answering, anything
  where `not` matters. Keep them, or remove a *shorter, safer* list that leaves negations in.

There is no universal answer. It depends on the job — which is the theme of this whole lab.

---
## Part 7 — Putting it together: a preprocessing pipeline

Real code wraps these steps into one function. Here is a typical pipeline for a
**search/topic** task, where removing stop words is the right call.

In [ ]:
import regex
from nltk.corpus import stopwords
sw = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()                          # 1. normalize case
    text = regex.sub(r"[^\w\s]", " ", text)      # 2. drop punctuation
    tokens = [t.lemma_ for t in nlp(text)]       # 3. tokenize + lemmatize (spaCy)
    tokens = [t for t in tokens if t.strip()]    # 4. drop blanks
    tokens = [t for t in tokens if t not in sw]  # 5. drop stop words
    return tokens

for r in reviews:
    print(preprocess(r))

# Verified output:
#   ['phone', 'absolutely', 'amazing', 'battery', 'life', 'bad']
#   ['I', 'like', 'camera', 'photo', 'blurry', 'colour', 'look', 'wash']
#   ['good', 'purchase', 'ever', 'study', 'show', 'last', 'long', 'competitor']
#   ['screen', 'crack', 'week', 'terrible', 'quality', 'would', 'recommend']

Each review is now a clean list of content words — lowercased, real lemmas, no
punctuation, no stop words. That is exactly the input the search and vector methods later
in this course expect.

But remember Part 6: this pipeline **removes `not`**, so it is wrong for sentiment. For a
sentiment task you would keep the stop words (or use a list that spares negation). The
pipeline is not one fixed thing — you assemble it from these steps to fit the job.

### Exercise 2

Write a **sentiment-safe** version, `preprocess_sentiment`, that does everything above
**except** stop-word removal — so `not` survives. Run it on the fourth review
(`"...would not recommend."`) and confirm `not` is still there.

In [ ]:
# YOUR CODE HERE
# Copy preprocess, delete the stop-word line. Run on reviews[3].

---
## Before you go

Check all of these:

- [ ] **Runtime → Restart and run all** works top to bottom with no errors
- [ ] You can tokenize with NLTK and spaCy, and say why `split()` is not enough
- [ ] You can explain the difference between **stemming** and **lemmatization**, and why
      lemmatization needs a part of speech
- [ ] You know why `organization -> organ` is fine for search but not for meaning
- [ ] You can explain **why removing stop words can be a serious mistake**
- [ ] You saved your own copy to Drive

**The one thing to carry forward:** every preprocessing step is optional and
task-dependent. Stemming, stop-word removal, even lowercasing — each throws information
away. Sometimes that is exactly right (search), sometimes it is a disaster (sentiment).
Choosing well is the skill.

**Next lab:** information retrieval — the term-document incidence matrix, and building an
inverted index to search a collection of documents fast.